# NOVALAINE : Customer 360 et segmentation RFM

Projet de pairagogie n°1, module Data Processing (M2 INSEEC 2026-2027), groupe 3.

Équipe : Nour El Manar KHEZZAR, Amira HAMDI, Pierre Quentin NGANDJUI TIAKO, Hayat AKBI.

Ce notebook porte tout le pipeline, de l'audit des exports bruts jusqu'aux tables finales. Il doit tourner de bout en bout après « Restart & Run All », en moins de 5 minutes, avec des chemins relatifs.

## Sommaire

0. Cadrage
1. Collecte et audit
2. Nettoyage et standardisation
3. Intégration et dédoublonnage des clients
4. Feature engineering : la vue client 360
5. Jeu de données prédictif et contrôles
6. Exploitation : la preuve d'usage
7. Industrialisation

Règles du brief à respecter partout : les fichiers de `sources_de_donnees/` ne sont jamais modifiés, toutes les sorties vont dans `sorties/`, aucune fonction n'utilise la date du jour, chaque règle de nettoyage indique combien de lignes elle touche.

Où enregistrer les sorties :

| Dossier | Contenu | Variable |
|---|---|---|
| `sorties/donnees/` | tables livrables (`clients_360.csv`, `dataset_rachat_6m.csv`, `table_correspondance_clients.csv`) | `DOSSIER_DONNEES` |
| `sorties/tableaux/` | tableaux en .csv (audit, réconciliation du CA, contrôles, volumétries), préfixés par la phase : `p1_audit.csv`, `p3_reconciliation_ca.csv`… | `DOSSIER_TABLEAUX` |
| `sorties/figures/` | graphes en .png, préfixés par la phase : `p6_ca_par_segment.png`… | `DOSSIER_FIGURES` |

Exemple : `df.to_csv(DOSSIER_TABLEAUX / "p1_audit.csv", index=False, encoding="utf-8-sig")` et `plt.savefig(DOSSIER_FIGURES / "p6_ca_par_segment.png", dpi=150, bbox_inches="tight")`.

## Paramètres et imports

Tous les paramètres du projet sont regroupés ici. On ne met aucune valeur en dur plus bas dans le notebook.

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Chemins (relatifs à la racine du dépôt)
DOSSIER_BRUT = Path("sources_de_donnees")       # exports bruts, jamais modifiés
DOSSIER_SORTIE = Path("sorties")
DOSSIER_DONNEES = DOSSIER_SORTIE / "donnees"     # tables livrables : clients_360.csv, dataset_rachat_6m.csv, table_correspondance_clients.csv
DOSSIER_TABLEAUX = DOSSIER_SORTIE / "tableaux"   # tableaux d'audit, de réconciliation, de contrôles, journal des volumes
DOSSIER_FIGURES = DOSSIER_SORTIE / "figures"     # graphes en .png pour le rapport et la soutenance
for dossier in (DOSSIER_DONNEES, DOSSIER_TABLEAUX, DOSSIER_FIGURES):
    dossier.mkdir(parents=True, exist_ok=True)

# Dates du projet (jamais today())
DATE_REFERENCE = pd.Timestamp("2026-09-30")    # date d'extraction, vue client 360
DATE_COUPURE = pd.Timestamp("2026-03-31")      # variables du jeu prédictif
DEBUT_CIBLE = pd.Timestamp("2026-04-01")       # fenêtre de la cible rachat_6m
FIN_CIBLE = pd.Timestamp("2026-09-30")

# Règles métier (fiche de cadrage, à confirmer avec le LISEZ-MOI)
FRAIS_LIVRAISON = 4.90            # web et marketplace
SEUIL_LIVRAISON_GRATUITE = 90.00  # après remises
TOLERANCE_EURO = 0.02             # écart accepté commande / somme des lignes

# Seuils de segmentation (fiche de cadrage)
JOURS_ACTIF = 180
JOURS_REACTIVATION = (181, 365)
JOURS_NOUVEAU_CLIENT = 90

GRAINE = 42
np.random.seed(GRAINE)

pd.set_option("display.max_columns", 50)

## Chargement des données brutes

Tout est lu en texte (`dtype=str`) pour ne perdre aucun format d'origine. Les conversions de types se font dans les phases 1 et 2.

In [ ]:
FICHIERS = {
    "clients": "clients.csv",
    "commandes": "commandes.csv",
    "lignes": "lignes_commande.csv",
    "produits": "produits.csv",
}

brut = {nom: pd.read_csv(DOSSIER_BRUT / fichier, dtype=str, keep_default_na=False)
        for nom, fichier in FICHIERS.items()}

journal_volumes = []  # une entrée par étape : (étape, table, lignes en entrée, lignes en sortie)

def tracer(etape, table, n_entree, n_sortie):
    """Ajoute une ligne au journal des volumétries et l'affiche."""
    journal_volumes.append({"etape": etape, "table": table,
                            "lignes_entree": n_entree, "lignes_sortie": n_sortie,
                            "ecart": n_sortie - n_entree})
    print(f"{etape:<45} {table:<10} {n_entree:>7} -> {n_sortie:>7}")

for nom, df in brut.items():
    tracer("Chargement", nom, len(df), len(df))

## Phase 0 : Cadrage

Objectif : Reformuler la commande de Camille Lefort et poser les définitions avant de coder.

À faire :
- [ ] Rappeler la demande en 5 lignes et les décisions qu'elle prendra avec nos tables
- [ ] Définitions retenues : client réel, commande valide, CA net, client actif, à réactiver, VIP, nouveau client
- [ ] Schéma du pipeline (entrées, étapes, sorties)

Livrable : fiche de cadrage d'une page (rendue : `docs/Rendu Projet - Cadrage - Groupe 3.docx`).

### Définitions retenues

| Notion | Définition retenue |
|---|---|
| Client réel | |
| Commande valide | |
| Chiffre d'affaires net | |
| Client actif | |
| Client à réactiver | |
| Client VIP | |
| Nouveau client | |

## Phase 1 : Collecte et audit

Objectif : Connaître précisément l'état des données avant de les modifier.

À faire :
- [ ] Types, volumétries, plages de dates, valeurs distinctes
- [ ] Grille d'audit de la séance 1 : complétude, validité, unicité, cohérence, conformité des formats, intégrité référentielle
- [ ] Chaque anomalie chiffrée (nombre et %) et classée par gravité et impact métier

Livrable : tableau d'audit chiffré (anomalie, fichier, colonne, volume, gravité, traitement envisagé).

Voir `docs/revue_audit_qualite.md` pour les points à reprendre de la première version (`Archives/`).

In [ ]:
# À compléter

## Phase 2 : Nettoyage et standardisation

Objectif : Rendre chaque colonne exploitable avec des règles explicites, appliquées en code.

À faire :
- [ ] Normaliser dates, montants, textes et modalités équivalentes (pays, canaux, statuts, consentement)
- [ ] Valider e-mails, codes postaux, dates de naissance
- [ ] Traiter les valeurs manquantes colonne par colonne (supprimer, imputer, signaler, laisser vide) et justifier
- [ ] Supprimer les doublons exacts de commandes, traiter les valeurs aberrantes (quantités, montants)

Livrable : fonctions de nettoyage commentées + journal des règles (règle, justification, volume touché).

In [ ]:
# À compléter

## Phase 3 : Intégration et dédoublonnage des clients

Objectif : Obtenir une personne = un identifiant, et rattacher toutes les commandes au bon client.

À faire :
- [ ] Clé de rapprochement des fiches en double (e-mail normalisé, nom, prénom, date de naissance) et justification
- [ ] Table de correspondance client_id vers client_unique_id, règle de survie des informations de contact
- [ ] Rattacher les commandes des fiches doublons, traiter les commandes orphelines
- [ ] Reconstruire les montants manquants ou invalides à partir des lignes, vérifier la cohérence commande / lignes

Livrable : `table_correspondance_clients.csv` + tableau de réconciliation du CA avant / après.

In [ ]:
# À compléter

## Phase 4 : Feature engineering : la vue client 360

Objectif : Calculer, pour chaque client réel, les variables utiles au marketing et à la prédiction.

À faire :
- [ ] Identité fiabilisée, RFM, comportement, rentabilité et retours, listes campagne
- [ ] Fonction `construire_features(commandes, clients, date_coupure)` réutilisable
- [ ] Scores R, F, M par quantiles (1 à 5) et règle de segmentation lisible par le métier

Livrable : `clients_360.csv` au 30/09/2026 + règle de segmentation documentée.

In [ ]:
def construire_features(commandes, clients, date_coupure):
    """Calcule une ligne par client réel avec les seules commandes antérieures ou égales à date_coupure."""
    # À compléter
    pass

In [ ]:
# À compléter

## Phase 5 : Jeu de données prédictif et contrôles

Objectif : Préparer le rachat à 6 mois sans fuite d'information, et prouver la qualité du résultat.

À faire :
- [ ] Variables reconstruites au 31/03/2026 avec les seules informations antérieures
- [ ] Cible rachat_6m : au moins une commande valide entre le 01/04/2026 et le 30/09/2026
- [ ] Au moins 8 contrôles automatiques (unicité des clients, pas de dates futures, somme des lignes = montant, aucune commande de la période cible dans les variables, etc.)

Livrable : `dataset_rachat_6m.csv` + rapport de contrôles qualité.

In [ ]:
controles = []  # une entrée par contrôle : nom, résultat, détail

def controler(nom, condition, detail=""):
    """Enregistre un contrôle qualité et son résultat."""
    controles.append({"controle": nom, "resultat": "OK" if bool(condition) else "ÉCHEC", "detail": detail})

# À compléter : au moins 8 contrôles

In [ ]:
# À compléter

## Phase 6 : Exploitation : la preuve d'usage

Objectif : Montrer à la direction CRM ce que la base propre permet de décider.

À faire :
- [ ] Trois listes cibles (réactivation, VIP, nouveaux clients) filtrées sur le consentement e-mail, avec leurs tailles
- [ ] 3 à 4 visuels : répartition des segments, CA par segment, poids des promotions, taux de retour par catégorie
- [ ] 3 recommandations CRM chiffrées

Livrable : une page « Ce que dit la base ».

In [ ]:
# À compléter

## Phase 7 : Industrialisation

Objectif : Permettre de relancer le traitement chaque trimestre sur un nouvel export.

À faire :
- [ ] Fonction `executer_pipeline(dossier_brut, dossier_sortie, date_coupure)`
- [ ] Volumétries journalisées à chaque étape
- [ ] Vérifier qu'un « Restart & Run All » reproduit exactement les mêmes tables

Livrable : notebook final propre + module `pipeline.py` (facultatif, valorisé).

In [ ]:
def executer_pipeline(dossier_brut, dossier_sortie, date_coupure):
    """Enchaîne toutes les étapes du chargement aux exports."""
    # À compléter
    pass

In [ ]:
# À compléter

## Journal des volumétries

Récapitulatif de toutes les étapes tracées avec `tracer()`.

In [ ]:
volumes = pd.DataFrame(journal_volumes)
volumes.to_csv(DOSSIER_TABLEAUX / "journal_volumes.csv", index=False, encoding="utf-8-sig")
volumes